# Validate Gaussian Splatting training on a real GPU

This notebook runs `worker/runner.py`'s `train_gaussian_splatting` for the first time on real CUDA hardware (Colab's free GPU). It reuses the SfM reconstruction already validated and committed in this repo (`experiments/sfm/data/synthetic_scene/`, 16/16 cameras registered, ~1.2% pose error vs. ground truth) so this notebook's only job is testing the previously-unverified training loop, not re-proving COLMAP works.

**Runtime > Change runtime type > GPU (T4 is fine)** before running these cells.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU attached -- go to Runtime > Change runtime type > GPU, then re-run.')

In [ ]:
!git clone https://github.com/yusupildan-wq/Scene-Reconstruction.git
%cd Scene-Reconstruction

In [ ]:
# torch is already installed (with CUDA) by Colab -- do not reinstall it here,
# a pip-installed torch could mismatch Colab's preconfigured CUDA setup.
!pip install -q pycolmap gsplat scipy

In [ ]:
import sys
sys.path.insert(0, 'worker')

import pycolmap
from pathlib import Path

recon = pycolmap.Reconstruction('experiments/sfm/data/synthetic_scene/sparse/0')
print(f'Loaded reconstruction: {recon.num_reg_images()} cameras, {recon.num_points3D()} points')

In [ ]:
from runner import SfmResult, train_gaussian_splatting

sfm = SfmResult(reconstruction=recon, images_dir=Path('experiments/sfm/data/synthetic_scene/images'))

# Fewer iterations than the default (3000) for a quick first sanity check --
# raise this once we know the loop runs correctly at all.
gaussians = train_gaussian_splatting(sfm, num_iterations=1000)
print('Trained', gaussians['means'].shape[0], 'Gaussians')

## Visual sanity check

Render the trained Gaussians from one of the known camera poses and compare
side-by-side against the real photo from that same pose -- if training worked,
these should look at least roughly similar (blurry/rough is expected at only
1000 iterations with no adaptive densification; this is checking "did it learn
anything at all," not final quality).

In [ ]:
import numpy as np
import torch
from PIL import Image
from gsplat import rasterization
from runner import _colmap_camera_to_K, _colmap_pose_to_viewmat
import matplotlib.pyplot as plt

device = torch.device('cuda')
image = next(iter(recon.images.values()))
real_photo = np.asarray(Image.open(sfm.images_dir / image.name).convert('RGB'))

means = torch.tensor(gaussians['means'], device=device)
quats = torch.tensor(gaussians['quats'], device=device)
scales = torch.tensor(gaussians['scales'], device=device)
opacities = torch.tensor(gaussians['opacities'], device=device)
colors = torch.tensor(gaussians['colors'], device=device)
viewmat = torch.tensor(_colmap_pose_to_viewmat(image), device=device).unsqueeze(0)
K = torch.tensor(_colmap_camera_to_K(recon.camera(image.camera_id)), device=device).unsqueeze(0)
h, w = real_photo.shape[:2]

with torch.no_grad():
    render, _alpha, _meta = rasterization(means, quats, scales, opacities, colors, viewmat, K, w, h, sh_degree=None)

# Contrast-stretch before display -- at only 1000 iterations the render is dim
# but has real content; a plain 0-1 clamp made it look solid black even though
# it wasn't (confirmed on a real run: render values ranged ~0.05-1.18, not
# near-zero -- this was a display issue, not a training bug).
rendered = (render[0] / render[0].max()).clamp(0, 1).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(real_photo); axes[0].set_title('Real photo'); axes[0].axis('off')
axes[1].imshow(rendered); axes[1].set_title('Trained Gaussian render (contrast-stretched)'); axes[1].axis('off')
plt.show()